##### Ideal QSVM (SVC Precomputed Kernel) - Lung Cancer

In [64]:
# To ensure reproducibility of results
from qiskit_machine_learning.utils import algorithm_globals
algorithm_globals.random_seed = 12345

In [65]:
# --- Libnray Imports ---
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, recall_score
from scipy.stats import chi2_contingency

In [66]:
# Qiskit Imports
# Definine quantum kernel
# Use the FidelityQuantumKernel class 

from qiskit.circuit.library import ZZFeatureMap
from qiskit.primitives import StatevectorSampler as Sampler
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel

In [67]:
# Load data first
lung_cancer_column_names = ['label'] + [f'attr_{i}' for i in range(1, 57)]
file_path_lung = r'C:\Users\User\Documents\MyProjects\FYP_ResearchProject\data\lung+cancer\lung-cancer.data'

# reads the data, treating "?" as missing values
df_lung = pd.read_csv(file_path_lung, header=None, names=lung_cancer_column_names, na_values=['?'])

print(f"Original shape of Lung Cancer data: {df_lung.shape}")

Original shape of Lung Cancer data: (32, 57)


In [68]:
# Mode imputation for missing values
modes = df_lung.mode().iloc[0]
df_lung.fillna(modes, inplace=True)

# Then check if all Nan are gone
print(f"Total missing values after imputation: {df_lung.isnull().sum().sum()}\n")

Total missing values after imputation: 0



In [69]:
# Target Binarization
df_lung['label_binary'] = df_lung['label'].apply(lambda x: 0 if x == 1 else 1)

In [70]:
# Separate Features & Target and Split Data
X_lung = df_lung.drop(['label', 'label_binary'], axis=1)
y_lung_binary = df_lung['label_binary']

In [ ]:
X_train_lc, X_test_lc, y_train_lc, y_test_lc = train_test_split(
    X_lung, y_lung_binary, test_size=0.3, random_state=42, stratify=y_lung_binary
)

print(f"Training samples: {X_train_lc.shape[0]}")
print(f"Test samples: {X_test_lc.shape[0]}\n")

In [ ]:
# One-Hot Encoding
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_train_lc_encoded = pd.DataFrame(
    encoder.fit_transform(X_train_lc),
    columns=encoder.get_feature_names_out()
)
X_test_lc_encoded = pd.DataFrame(
    encoder.transform(X_test_lc),
    columns=encoder.get_feature_names_out()
)

In [73]:
# Feature Selection - Cramer's V
def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    if min((kcorr-1), (rcorr-1)) == 0: return 0
    return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))

cramers_scores = {col: cramers_v(X_train_lc_encoded[col], y_train_lc) for col in X_train_lc_encoded.columns}
cramers_series = pd.Series(cramers_scores).sort_values(ascending=False)

N_FEATURES_TO_SELECT = 10 
top_features = cramers_series.head(N_FEATURES_TO_SELECT).index.tolist()

X_train_lc_final = X_train_lc_encoded[top_features]
X_test_lc_final = X_test_lc_encoded[top_features]

print("--- Data Preprocessing Complete ---")
print(f"Final training data shape: {X_train_lc_final.shape}")
print(f"Final testing data shape: {X_test_lc_final.shape}\n")


--- Data Preprocessing Complete ---
Final training data shape: (22, 10)
Final testing data shape: (10, 10)



##### Quantum Kernel Implementation

In [ ]:
# ===================================================================
# OPTIMIZATION EXPERIMENT CONFIGURATION
# ===================================================================
print("=" * 70)
print("Setting up Quantum Kernel (Ideal - Statevector)")
print("=" * 70)

# Feature map
fm = ZZFeatureMap(feature_dimension=N_FEATURES_TO_SELECT, reps=1, entanglement='linear')

# Ideal sampler (statevector - no noise)
sampler = Sampler(default_shots=256)

# Fidelity and kernel
fidelity = ComputeUncompute(sampler=sampler)
qkernel = FidelityQuantumKernel(fidelity=fidelity, feature_map=fm)

print("Quantum kernel ready (ZZFeatureMap, reps=1, 256 shots)\n")

Setting up Quantum Kernel (Ideal - Statevector)
Quantum kernel ready (ZZFeatureMap, reps=1, 256 shots)



##### Compute Kernel Matrices

In [75]:
# Compute Kernel Matrices (once, upfront)
print("Computing kernel matrices...")
start_kernel = time.time()

matrix_train_lc = qkernel.evaluate(x_vec=X_train_lc_final.to_numpy())
matrix_test_lc = qkernel.evaluate(x_vec=X_test_lc_final.to_numpy(), y_vec=X_train_lc_final.to_numpy())

kernel_time = time.time() - start_kernel
print(f"Kernel matrices computed in {kernel_time:.2f} seconds.")

Computing kernel matrices...
Kernel matrices computed in 4.03 seconds.


In [76]:
# ===================================================================
# Ideal QSVM Implementation
# ===================================================================

print("=" * 70)
print("IDEAL QSVM IMPLEMENTATION")
print("=" * 70)

# --- Compute Kernel Matrices ---
print("\nComputing kernel matrices...")
start_kernel = time.time()

matrix_train = qkernel.evaluate(x_vec=X_train_lc_final)
matrix_test = qkernel.evaluate(x_vec=X_test_lc_final, y_vec=X_train_lc_final)

kernel_time = time.time() - start_kernel
print(f"Kernel matrices computed in {kernel_time:.2f} seconds.")

IDEAL QSVM IMPLEMENTATION

Computing kernel matrices...
Kernel matrices computed in 3.97 seconds.


In [77]:
# Save Kernel Matrices
print("\nSaving computed kernel matrices...")
np.save('matrix_train_ideal.npy', matrix_train)
np.save('matrix_test_ideal.npy', matrix_test)
print(f"Saved: matrix_train_ideal.npy ({matrix_train.shape})")
print(f"Saved: matrix_test_ideal.npy ({matrix_test.shape})")


Saving computed kernel matrices...
Saved: matrix_train_ideal.npy ((22, 22))
Saved: matrix_test_ideal.npy ((10, 22))


##### Training with GridSearchCV

In [78]:
# --- Grid Search ---
print("\nStarting Grid Search...")

# param_grid = {'C': [0.1, 1, 10, 100]}
param_grid = {'C': [0.001, 0.01, 0.1, 1, 10, 100]}
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    SVC(kernel='precomputed', class_weight='balanced'),
    param_grid,
    cv=cv,
    scoring='accuracy',
    verbose=1,
    n_jobs=-1
)

start_time = time.time()
grid_search.fit(matrix_train, y_train_lc)
qsvc_model = grid_search.best_estimator_
train_time = time.time() - start_time

print(f"Best C parameter: {grid_search.best_params_['C']}")


Starting Grid Search...
Fitting 3 folds for each of 6 candidates, totalling 18 fits
Best C parameter: 100


In [79]:
# --- Overall Model Evaluation ---
y_train_pred = qsvc_model.predict(matrix_train)
y_test_pred = qsvc_model.predict(matrix_test)

train_accuracy = accuracy_score(y_train_lc, y_train_pred)
test_accuracy = accuracy_score(y_test_lc, y_test_pred)
recall_spam = recall_score(y_test_lc, y_test_pred, pos_label=1)
generalization_gap = abs(train_accuracy - test_accuracy)

print("\n" + "=" * 70)
print("Ideal QSVM on (Baseline)")
print("=" * 70)
print(f"Samples Used: {X_train_lc_final.shape[0]} train, {X_test_lc.shape[0]} test")
print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Spam Recall (Class 1): {recall_spam:.4f}")
print(f"Generalization Gap: {generalization_gap:.4f}")
print(f"Training Time: {train_time:.2f} seconds")
print(f"Kernel Computation Time: {kernel_time:.2f} seconds")

print("\nClassification Report (Test Set):")
print(classification_report(y_test_lc, y_test_pred, zero_division=0))


Ideal QSVM on (Baseline)
Samples Used: 22 train, 10 test
Training Accuracy: 0.9545
Test Accuracy: 0.5000
Spam Recall (Class 1): 0.7143
Generalization Gap: 0.4545
Training Time: 0.03 seconds
Kernel Computation Time: 3.97 seconds

Classification Report (Test Set):
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.62      0.71      0.67         7

    accuracy                           0.50        10
   macro avg       0.31      0.36      0.33        10
weighted avg       0.44      0.50      0.47        10

